In [161]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import OrderedDict

In [35]:
import torchvision
import torchvision.transforms as T

In [122]:
device = 'cpu'

In [13]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.mps.is_available() else 'cpu')

In [123]:
class Add(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        return x+y

In [412]:
class Attention(nn.Module):
    def __init__(self, hidden_dim, nb_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(hidden_dim, nb_heads, batch_first=True)

    def forward(self, x):
        return self.attention(x,x,x, need_weights=False)[0]

In [413]:
def pos_encoding(d, sl):
    pos = torch.arange(0,sl,device=device).view(sl,1)
    freq = 1 / 10000 ** (torch.arange(0, d//2,device=device) * 2 / d)
    y = pos*freq
    s = y.sin()
    c = y.cos()
    return torch.stack([s,c], dim=2).view(sl, d)

In [438]:
class Patch(nn.Module):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def forward(self, x):
        B,C,H,W = x.shape
        P = self.patch_size
        N = H*W//(P**2)
        x = x.reshape(B,C,H//P,P,W//P,P)
        x = x.permute(0,2,4,1,3,5)
        x = x.reshape(B,N,-1)
        return x

In [451]:
class ViT(nn.Module):
    def __init__(self, hidden_dim, patch_size, channels, nb_heads, nb_layers, height, width):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.patch_size = patch_size
        self.channels = channels
        self.height = height
        self.width = width
        self.embedding = nn.Linear(patch_size**2*channels,hidden_dim)
        self.xclass = nn.Parameter(torch.randn(hidden_dim))
        n = height*width // (patch_size**2)
        self.posenc = nn.Parameter(torch.randn(n+1,hidden_dim))
        self.patch = Patch(patch_size)
        self.layers = nn.ModuleList([
             nn.ModuleDict({
                'norm1': nn.LayerNorm(hidden_dim),
                'att': Attention(hidden_dim, nb_heads),
                'norm2': nn.LayerNorm(hidden_dim),
                'ffn': nn.Sequential(nn.Linear(hidden_dim, 4*hidden_dim), nn.GELU(), nn.Linear(4*hidden_dim, hidden_dim)),
             }) for _ in range(nb_layers)
        ])
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.last_layer_mlp = nn.Linear(hidden_dim, hidden_dim)
        self.batch_norm = nn.BatchNorm1d(hidden_dim)
    
    def forward(self, x):
        # x.shape = bs * c * height * weight
        p = self.patch_size
        c = self.channels
        n = x.shape[2]*x.shape[3]//(p**2)
        x = self.patch(x)
        pos_enc = pos_encoding(self.hidden_dim, n+1)
        x = torch.concat([self.xclass.expand(x.shape[0], 1, -1), self.embedding(x)], dim=-2)+pos_enc
        for layer in self.layers:
            x = layer['att'](layer['norm1'](x))+x
            x = layer['ffn'](layer['norm2'](x))+x
        return self.batch_norm(self.last_layer_mlp(self.layer_norm(x)[:,0,:]))
        

In [452]:
# with a patch size of 14, 12 layers, 3 attention heads, and hidden dimensions of 192
vit = ViT(patch_size=14, nb_layers=12, nb_heads=3, hidden_dim=192, channels=3, height=224, width=224)

In [453]:
images = torch.rand(10, 3, 224,224)*255

In [454]:
transform = T.ToPILImage()

In [455]:
vit(images)

tensor([[ 0.9129, -0.9429,  0.8657,  ..., -0.5008, -1.1229,  1.1051],
        [ 1.0005,  1.5055,  0.1429,  ...,  1.3553,  1.1951,  0.6581],
        [-0.1466,  0.5875, -0.4382,  ..., -0.2913,  0.3578, -0.1887],
        ...,
        [-1.3095,  0.4216,  1.1482,  ..., -0.0525, -0.0250, -0.8400],
        [-0.2657, -0.3376,  0.3346,  ...,  0.3506, -2.0840,  0.7990],
        [ 1.6139,  0.5536, -1.8965,  ...,  1.1813, -0.4224,  1.7692]],
       grad_fn=<NativeBatchNormBackward0>)